In [ ]:
!pip install transformers --upgrade

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
import torch
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler
import torch.nn.functional as F
from tqdm import tqdm
import sklearn.utils
import json
from collections import defaultdict
import matplotlib.pyplot as plt
# os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def clean_text(text):
    if isinstance(text, str):
        return re.sub(r'[^a-zA-Z0-9\s]', '', text.lower())
    elif isinstance(text, np.ndarray):
        return np.array([clean_text(t) for t in text])
    else:
        raise ValueError("Unsupported type")

In [ ]:
labeled_data = pd.read_csv("/home/jovyan/work/eq5d/eq-5d-projects/eq-5d-dynamic-lr/data/eq-5d-200-records.tsv", sep="\t")
unlabeled_data = pd.read_csv("/home/jovyan/work/eq5d/eq-5d-projects/eq-5d-dynamic-lr/data/random_200_records.csv")

for col in ['Title', 'Abstract', 'Keywords']:
    labeled_data[col] = labeled_data[col].apply(clean_text)
    unlabeled_data[col] = unlabeled_data[col].apply(clean_text)

labeled_data['combined_text'] = labeled_data['Title'] + ' [SEP] ' + labeled_data['Keywords'] + ' [SEP] ' + labeled_data['Abstract']
unlabeled_data['combined_text'] = unlabeled_data['Title'] + ' [SEP] ' + unlabeled_data['Keywords'] + ' [SEP] ' + unlabeled_data['Abstract']


In [ ]:
abstractslbl = labeled_data['Abstract']
titleslbl = labeled_data['Title']
keywordsslbl = labeled_data['Keywords']
lbls = labeled_data['Label']

abstractsunlbl = unlabeled_data['Abstract']
titlesunlbl = unlabeled_data['Title']
keywordsunlbl = unlabeled_data['Keywords']

In [ ]:
print(np.shape(labeled_data))
print(np.shape(unlabeled_data))

In [ ]:
labels = np.unique(lbls, return_counts=True)[0]
labelCounts=  np.unique(lbls, return_counts=True)[1]

print("Labels ", labels)
print("Label counts ", labelCounts)

In [ ]:
labeled_data_shuffled = sklearn.utils.shuffle(labeled_data)
abstracts_shuffled = labeled_data_shuffled['Abstract'].values
keywords_shuffled = labeled_data_shuffled['Keywords'].values
domains_shuffled = labeled_data_shuffled['Label'].values
allLabels_shuffled = labeled_data_shuffled['No'].values
allLabels_digit_shuffled = allLabels_shuffled.astype(int)

In [ ]:
print(np.shape(labeled_data_shuffled))
print(np.shape(abstracts_shuffled))
print(np.shape(keywords_shuffled))
print(np.shape(domains_shuffled))

print(labeled_data_shuffled['combined_text'][0])

In [ ]:
def encode_data(data, tokenizer, max_length=128, labeled=True):
    inputs = tokenizer(
        data['combined_text'].tolist(),
        max_length=max_length,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    if labeled:
        labels = torch.tensor(data['Label'].values.astype(int))
        return TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
    else:
        return TensorDataset(inputs['input_ids'], inputs['attention_mask'])

In [ ]:
train_data, test_data = train_test_split(labeled_data_shuffled, test_size=0.3, random_state=42, stratify=labeled_data_shuffled['Label'])
_, val_data = train_test_split(test_data, test_size=0.5, random_state=42, stratify=test_data['Label'])

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
biobert_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

In [ ]:
num_labels=len(labeled_data['Label'].unique())
print(num_labels)

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(labeled_data['Label'].unique()))
biobert_model = AutoModelForSequenceClassification.from_pretrained("dmis-lab/biobert-base-cased-v1.2", num_labels=len(labeled_data['Label'].unique()))

In [ ]:
bert_train_loader = DataLoader(encode_data(train_data, bert_tokenizer), sampler=RandomSampler(train_data), batch_size=32)
bert_val_loader = DataLoader(encode_data(val_data, bert_tokenizer), batch_size=32)
bert_test_loader = DataLoader(encode_data(test_data, bert_tokenizer), batch_size=32)

biobert_train_loader = DataLoader(encode_data(train_data, biobert_tokenizer), sampler=RandomSampler(train_data), batch_size=32)
biobert_val_loader = DataLoader(encode_data(val_data, biobert_tokenizer), batch_size=32)
biobert_test_loader = DataLoader(encode_data(test_data, biobert_tokenizer), batch_size=32)

In [ ]:
print(train_data.shape)
print(val_data.shape)
print(test_data.shape)

In [ ]:
def train_model(model, train_loader, val_loader, save_path, epochs=20, learning_rates=[2e-5, 5e-6, 1e-6, 2e-6], patience=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    best_val_f1 = 0
    patience_counter = 0

    for lr in learning_rates:
        print(f"\nTraining with LR={lr}")
        optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
        total_steps = len(bert_train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0.1 * total_steps, num_training_steps=total_steps)
        # scheduler = get_linear_schedule_with_warmup(
        #     optimizer, num_warmup_steps=0.1*len(train_loader)*epochs,
        #     num_training_steps=len(train_loader)*epochs
        #)

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", unit="batch"):
                batch = tuple(t.to(device) for t in batch)
                input_ids, input_mask, labels = batch
                model.zero_grad()
                outputs = model(input_ids=input_ids, attention_mask=input_mask, labels=labels)
                loss = outputs.loss
                loss.backward()
                total_loss += loss.item()
                optimizer.step()
                scheduler.step()
            avg_train_loss = total_loss / len(train_loader)
            print(f"Epoch {epoch+1} Loss: {avg_train_loss:.4f}")

            # Validation
            model.eval()
            preds, true = [], []
            for batch in val_loader:
                batch = tuple(t.to(device) for t in batch)
                input_ids, input_mask, labels = batch
                with torch.no_grad():
                    outputs = model(input_ids=input_ids, attention_mask=input_mask)
                logits = outputs.logits.detach().cpu().numpy()
                batch_predictions = np.argmax(logits, axis=1)
                preds.extend(batch_predictions)
                true.extend(labels.to('cpu').numpy())
            f1 = f1_score(true, preds, average='micro')
            print(f"Validation Micro F1: {f1:.4f}")
            # val_micro_f1 = f1_score(true_labels, predictions, average='micro')
            print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {avg_train_loss}, Val Micro F1: {f1}")

            if f1 > best_val_f1:
                best_val_f1 = f1
                patience_counter = 0
                torch.save(model.state_dict(), save_path)
                print("Best model saved.")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print("Early stopping.")
                    break

In [ ]:
train_model(bert_model, bert_train_loader, bert_val_loader, 'bert_best_model.pth')
train_model(biobert_model, biobert_train_loader, biobert_val_loader, 'biobert_best_model.pth')

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    preds, true = [], []
    for batch in loader:
        batch = tuple(t.to(device) for t in batch)
        with torch.no_grad():
            logits = model(input_ids=batch[0], attention_mask=batch[1]).logits
        preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
        true.extend(batch[2].cpu().numpy())

    print("Confusion Matrix:")
    print(confusion_matrix(true, preds))
    print("Classification Report:")
    print(classification_report(true, preds))
    print(f"Macro F1: {f1_score(true, preds, average='macro'):.4f}")
    print(f"Micro F1: {f1_score(true, preds, average='micro'):.4f}")
    print(f"Weighted F1: {f1_score(true, preds, average='weighted'):.4f}")

In [ ]:
bert_model.load_state_dict(torch.load('bert_best_model.pth'))
biobert_model.load_state_dict(torch.load('biobert_best_model.pth'))

In [ ]:
print("\nEvaluating bert:")
evaluate_model(bert_model, bert_test_loader)

print("\nEvaluating BioBERT:")
evaluate_model(biobert_model, biobert_test_loader)

In [ ]:
def pseudo_label_prediction(model, loader):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    predictions, probabilities = [], []
    for batch in loader:
        batch = tuple(t.to(device) for t in batch)
        input_ids, input_mask = batch  # Expecting only 2 items
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=input_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1).cpu().numpy()
        batch_predictions = np.argmax(probs, axis=1)
        predictions.extend(batch_predictions)
        probabilities.extend(probs.max(axis=1))  # max probability, predicted label

    return predictions, probabilities

In [ ]:
unlabeled_data_shuffled = sklearn.utils.shuffle(unlabeled_data)

In [ ]:
def cross_label_unlabeled_data(bert_model, biobert_model, unlabeled_data, bert_tokenizer, bioberttokenizer, threshold=0.9):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    bert_unlabeled_dataset = encode_data(unlabeled_data, bert_tokenizer, labeled=False)
    biobert_unlabeled_dataset = encode_data(unlabeled_data, biobert_tokenizer, labeled=False)

    bert_loader = DataLoader(bert_unlabeled_dataset, batch_size=32)
    biobert_loader = DataLoader(biobert_unlabeled_dataset, batch_size=32)

    bert_preds, bert_probs = pseudo_label_prediction(bert_model, bert_loader)
    biobert_preds, biobert_probs = pseudo_label_prediction(biobert_model, biobert_loader)

    bert_for_biobert = unlabeled_data.copy()
    bert_for_biobert['Label'] = bert_preds
    bert_for_biobert['Confidence'] = bert_probs
    bert_for_biobert = bert_for_biobert[bert_for_biobert['Confidence'] >= threshold]

    biobert_for_bert = unlabeled_data.copy()
    biobert_for_bert['Label'] = biobert_preds
    biobert_for_bert['Confidence'] = biobert_probs
    biobert_for_bert = biobert_for_bert[biobert_for_bert['Confidence'] >= threshold]

    return bert_for_biobert, biobert_for_bert


In [ ]:
def co_training_loop(bert_model, biobert_model, labeled_data, unlabeled_data, bert_tokenizer, biobert_tokenizer, num_iterations=3, threshold=0.9):
   
    for iteration in range(num_iterations):
        print(f"\n========== Co-Training Iteration {iteration + 1} ==========")

        # ########### Cross-label high-confidence predictions
        bert_for_biobert, biobert_for_bert = cross_label_unlabeled_data(
            bert_model, biobert_model,
            unlabeled_data, bert_tokenizer, biobert_tokenizer,
            threshold=threshold
        )

        print(f"Added {len(bert_for_biobert)} new samples to BioBERT training.")
        print(f"Added {len(biobert_for_bert)} new samples to bert training.")

        # Combine with original labeled data
        biobert_combined = pd.concat([labeled_data, bert_for_biobert], ignore_index=True)
        bert_combined = pd.concat([labeled_data, biobert_for_bert], ignore_index=True)

        # Split and encode
        biobert_train, biobert_test = train_test_split(biobert_combined, test_size=0.3, stratify=biobert_combined['Label'], random_state=42)
        _, biobert_val = train_test_split(biobert_test, test_size=0.5, stratify=biobert_test['Label'], random_state=42)

        bert_train, bert_test = train_test_split(bert_combined, test_size=0.3, stratify=bert_combined['Label'], random_state=42)
        _, bert_val = train_test_split(bert_test, test_size=0.5, stratify=bert_test['Label'], random_state=42)

        biobert_train_loader = DataLoader(encode_data(biobert_train, biobert_tokenizer), sampler=RandomSampler(biobert_train), batch_size=32)
        biobert_val_loader = DataLoader(encode_data(biobert_val, biobert_tokenizer), batch_size=32)

        bert_train_loader = DataLoader(encode_data(bert_train, bert_tokenizer), sampler=RandomSampler(bert_train), batch_size=32)
        bert_val_loader = DataLoader(encode_data(bert_val, bert_tokenizer), batch_size=32)

        # Retrain both models
        train_model(biobert_model, biobert_train_loader, biobert_val_loader, f'biobert_iter{iteration + 1}.pth')
        train_model(bert_model, bert_train_loader, bert_val_loader, f'bert_iter_2{iteration + 1}.pth')

        print("\nEvaluating BioBERT:")
        evaluate_model(biobert_model, biobert_val_loader)

        print("\nEvaluating bert:")
        evaluate_model(bert_model, bert_val_loader)


In [ ]:
co_training_loop(
    bert_model, biobert_model,
    labeled_data=labeled_data,
    unlabeled_data=unlabeled_data_shuffled,
    bert_tokenizer=bert_tokenizer,
    biobert_tokenizer=biobert_tokenizer,
    num_iterations=3,
    threshold=0.9
)


In [ ]:
def get_model_predictions(unlabeled_data, model, tokenizer):
    dataset = encode_data(unlabeled_data, tokenizer, labeled=False)
    loader = DataLoader(dataset, batch_size=32)

    preds, probs = pseudo_label_prediction(model, loader)
    return preds, probs


In [ ]:
bert_preds, bert_probs = get_model_predictions(unlabeled_data, bert_model, bert_tokenizer)
biobert_preds, biobert_probs = get_model_predictions(unlabeled_data, biobert_model, biobert_tokenizer)


In [ ]:
final_unlabeled = unlabeled_data.copy()
final_unlabeled['bert_Pred'] = bert_preds
final_unlabeled['bert_Prob'] = bert_probs
final_unlabeled['biobert_Pred'] = biobert_preds
final_unlabeled['biobert_Prob'] = biobert_probs

# choose prediction from model with higher confidence
final_unlabeled['Best_Pred'] = [
    s if sp >= rp else r
    for s, r, sp, rp in zip(bert_preds, biobert_preds, bert_probs, biobert_probs)
]

final_unlabeled['Best_Model'] = [
    'bert' if sp >= rp else 'biobert'
    for sp, rp in zip(bert_probs, biobert_probs)
]


In [ ]:
final_unlabeled.to_csv("final_bert_biobert_unlabeled_predictions_best_model.csv", index=False)


In [ ]:
# print(np.shape(predictions))
# print(np.shape(probabilities))
print(labeled_data.columns)
print(final_unlabeled.columns)

In [ ]:
unlabeled_data = final_unlabeled.rename(columns={'N': 'No', 'Best_Pred': 'Label'})

labeled_subset = labeled_data[['No', 'Label', 'Title', 'Abstract', 'Keywords', 'combined_text']]
unlabeled_subset = unlabeled_data[['No', 'Label', 'Title', 'Abstract', 'Keywords', 'combined_text']]

combined_data = pd.concat([labeled_subset, unlabeled_subset], ignore_index=True)

print(combined_data.head())

In [ ]:
print(np.shape(combined_data))

In [ ]:
bert_train_data_new, bert_test_data_new = train_test_split(combined_data, test_size=0.3, random_state=42, stratify=combined_data['Label'])
_, bert_val_data_new = train_test_split(bert_test_data_new, test_size=0.5, random_state=42, stratify=bert_test_data_new['Label'])

In [ ]:
bert_train_dataset_new = encode_data(bert_train_data_new, bert_tokenizer)
bert_val_dataset_new = encode_data(bert_val_data_new, bert_tokenizer)
bert_test_dataset_new = encode_data(bert_test_data_new, bert_tokenizer)

In [ ]:
bert_train_dataloader_new = DataLoader(bert_train_dataset_new, sampler=RandomSampler(bert_train_dataset_new), batch_size=32)
bert_val_dataloader_new = DataLoader(bert_val_dataset_new, sampler=SequentialSampler(bert_val_dataset_new), batch_size=32)
bert_test_dataloader_new = DataLoader(bert_test_dataset_new, sampler=SequentialSampler(bert_test_dataset_new), batch_size=32)

In [ ]:
train_model(bert_model, bert_train_dataloader_new, bert_val_dataloader_new, 'bert_best_model_2_400.pth')

In [ ]:
bert_model.load_state_dict(torch.load('bert_best_model_2_400.pth'))


In [ ]:
print("\nEvaluating bert with Validation set - 400:")
evaluate_model(bert_model, bert_val_dataloader_new)

In [ ]:
print("\nEvaluating bert with Test set - 400:")
evaluate_model(bert_model, bert_test_dataloader_new)

In [ ]:
print("\nEvaluating bert - Original Test set 200:")
evaluate_model(bert_model, bert_test_loader)

In [ ]:
print(np.shape(combined_data))

In [ ]:
biobert_train_data_new, biobert_test_data_new = train_test_split(combined_data, test_size=0.3, random_state=42, stratify=combined_data['Label'])
_, biobert_val_data_new = train_test_split(biobert_test_data_new, test_size=0.5, random_state=42, stratify=biobert_test_data_new['Label'])

In [ ]:
biobert_train_dataset_new = encode_data(biobert_train_data_new, biobert_tokenizer)
biobert_val_dataset_new = encode_data(biobert_val_data_new, biobert_tokenizer)
biobert_test_dataset_new = encode_data(biobert_test_data_new, biobert_tokenizer)

In [ ]:
biobert_train_dataloader_new = DataLoader(biobert_train_dataset_new, sampler=RandomSampler(biobert_train_dataset_new), batch_size=32)
biobert_val_dataloader_new = DataLoader(biobert_val_dataset_new, sampler=SequentialSampler(biobert_val_dataset_new), batch_size=32)
biobert_test_dataloader_new = DataLoader(biobert_test_dataset_new, sampler=SequentialSampler(biobert_test_dataset_new), batch_size=32)


In [ ]:
train_model(biobert_model, biobert_train_dataloader_new, biobert_val_dataloader_new, 'biobert_best_model_2_400.pth')

In [ ]:
biobert_model.load_state_dict(torch.load('biobert_best_model_2_400.pth'))

In [ ]:
print("\nEvaluating biobert with Validation set - 400:")
evaluate_model(biobert_model, biobert_val_dataloader_new)

In [ ]:
print("\nEvaluating biobert with Test set - 400:")
evaluate_model(biobert_model, biobert_test_dataloader_new)

In [ ]:
print("\nEvaluating biobert - Original Test set 200:")
evaluate_model(biobert_model, biobert_test_loader)

In [ ]:
train_model(bert_model, bert_train_dataloader_new, bert_val_loader, 'bert_best_model_vlidation_with_original_2_200.pth')
train_model(biobert_model, biobert_train_dataloader_new, biobert_val_loader, 'biobert_best_model_vlidation_with_original_2_200.pth')

In [ ]:
bert_model.load_state_dict(torch.load('bert_best_model_vlidation_with_original_2_200.pth'))
biobert_model.load_state_dict(torch.load('biobert_best_model_vlidation_with_original_2_200.pth'))

In [ ]:
print("\nEvaluating bert with Validation set - 400:")
evaluate_model(bert_model, bert_val_dataloader_new)

In [ ]:
print("\nEvaluating bert with Test set - 400:")
evaluate_model(bert_model, bert_test_dataloader_new)

In [ ]:
print("\nEvaluating bert - Original Test set 200:")
evaluate_model(bert_model, bert_test_loader)

In [ ]:
print("\nEvaluating biobert with Validation set - 400:")
evaluate_model(biobert_model, biobert_val_dataloader_new)

In [ ]:
print("\nEvaluating biobert with Test set - 400:")
evaluate_model(biobert_model, biobert_test_dataloader_new)

In [ ]:
print("\nEvaluating biobert - Original Test set 200:")
evaluate_model(biobert_model, biobert_test_loader)